**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Préparation des points de collecte de donnees sols pour les parcelles avec arbres

# Création du Fichier `typeDeSolParZH.shp` : Points de collecte

## 1. Description du projet

Ce notebook est la première étape du traitement des données de sol. Son rôle est de préparer une liste propre et géolocalisée de **points d'échantillonnage** à partir du parcellaire enrichi. Il s'assure que chaque point (qu'il représente une zone d'influence d'arbre ou une parcelle classique) possède la géométrie la plus pertinente et est enrichi avec les informations de classification initiales (type de sol, type de champ, présence d'arbre).

---
## 2. Objectifs

* **Charger** le parcellaire enrichi (`.shp`) et le fichier CSV contenant le type de champ.
* **Nettoyer** les données en gérant les doublons et les non-concordances.
* **Fusionner** les deux sources de données via une jointure.
* **Créer** une couche de points unifiée en appliquant un **traitement géométrique différencié** (coordonnées réelles pour les arbres, centroïdes pour les parcelles).
* **Construire** la variable de classification finale **`ZONE_PEDO`**.
* **Exporter** la liste finale des points d'échantillonnage au format CSV pour le notebook suivant.

---
## 3. Fichiers en Entrée et en Sortie

### 3.1. Fichiers en Entrée
* **Parcellaire Enrichi :** `data/sols/shapefiles/raw/Parcellaire_Arbre_Carbone.shp`
* **Type de Champ :** `data/sols/csv/raw/malou_0_30.csv`

### 3.2. Fichier en Sortie
* **Points d'échantillonnage :** `data/sols/csv/processed/points_échantillonnage_complets.csv`
---

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from typing import List, Union

In [68]:
# Définition du chemin de base du projet
base_dir = Path.cwd().parent.resolve()

# Chemin vers le shapefile AVEC arbres
shp_input_path = base_dir / "data" / "sols" / "shapefiles" / "raw" / "Parcellaire_Arbre_Carbone.shp"

# Chemin pour le fichier CSV de sortie final
output_points_path = base_dir / "data" / "sols" / "csv" / "processed" / "points_echantillonnage_complets.csv"

print(f"Chemin du fichier d'entrée : {shp_input_path}")

Chemin du fichier d'entrée : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\shapefiles\raw\Parcellaire_Arbre_Carbone.shp


In [69]:
# ----------------- CHARGEMENT ----------------- #

try:
    # Charger le shapefile dans un GeoDataFrame
    gdf_parcelles = gpd.read_file(shp_input_path)
    
    # Afficher un message de succès et des informations sur le DataFrame
    print(f"\n✅ Fichier '{shp_input_path.name}' chargé avec succès.")
    print(f"   Dimensions du tableau (lignes, colonnes) : {gdf_parcelles.shape}")
    print(f"   Nombre de parcelles (lignes) : {gdf_parcelles.shape[0]}")
    print(f"   Nombre d'attributs (colonnes) : {gdf_parcelles.shape[1]}")
    
    print("\nAperçu des 5 premières lignes :")
    display(gdf_parcelles.head())

    # --- LIGNES AJOUTÉES --- #
    # Vérifier que la colonne 'Arbre' existe avant de l'utiliser
    if 'Arbre' in gdf_parcelles.columns:
        print("\n📊 Statistiques pour la colonne 'Arbre' :")
        
        # Compter le nombre d'occurrences de chaque valeur (0 et 1)
        comptage_arbres = gdf_parcelles['Arbre'].value_counts()
        display(comptage_arbres)
        
        # Afficher spécifiquement le nombre de parcelles sans arbres
        nombre_sans_arbre = comptage_arbres.get(0, 0) # .get(0, 0) pour éviter une erreur si aucune parcelle n'a la valeur 0
        print(f"\n   Nombre de parcelles SANS arbre ('Arbre' == 0) : {nombre_sans_arbre}")
    else:
        print("\n❌ ATTENTION : La colonne 'Arbre' est manquante, impossible de faire le décompte.")
    # ------------------------- #

except Exception as e:
    print(f"\nERREUR lors du chargement du fichier : {e}")


✅ Fichier 'Parcellaire_Arbre_Carbone.shp' chargé avec succès.
   Dimensions du tableau (lignes, colonnes) : (751, 29)
   Nombre de parcelles (lignes) : 751
   Nombre d'attributs (colonnes) : 29

Aperçu des 5 premières lignes :


,Id,N°_PARCEL,N°_FOYER,SAISON,NOM_UTILIS,NOM_PROPRI,UTL_2012,NOM_CHAMPS,DIST_ENQ,DIST_CARTO,...,Surface_Ar,Dist_Centr,X_Centroid,Y_Centroid,Stock_C,StockC_0_3,parcel_id,Surface__1,Dist_Cen_1,geometry
0,0,147.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Pifind,1.0,0.0,...,226.463297,48.373962,337365.773977,1.603611e+06,9.22,17.44,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,3181.224210,17.652819,"MULTIPOLYGON (((337434.305 1603598.415, 337435..."
1,0,147.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Pifind,1.0,0.0,...,0.000000,0.000000,337365.773977,1.603611e+06,9.22,17.44,c6c85252-0ee6-44b2-97bf-99724c6dd923,0.000000,0.000000,"POLYGON ((337438.043 1603638.866, 337438.614 1..."
2,0,149.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Ngolsima,2.0,0.0,...,171.266369,51.310047,336099.653300,1.603497e+06,6.60,10.45,dae3c8e5-9f33-4354-826d-aed699b6c17c,1045.516931,46.730772,"MULTIPOLYGON (((336159.144 1603477.128, 336160..."
3,0,149.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Ngolsima,2.0,0.0,...,0.000000,0.000000,336099.653300,1.603497e+06,6.60,10.45,c33f3753-34ef-4b61-ba17-3cc4fe29bbd3,0.000000,0.000000,"POLYGON ((336165.254 1603521.577, 336167.671 1..."
4,0,150.0,15.0,1.0,Marie DIOUF,Marie DIOUF,0.0,Maygueran,2.0,0.0,...,874.964689,60.283415,336302.866304,1.603039e+06,6.44,9.38,e85b0517-6fda-4422-95a3-571360932f23,2503.005594,8.629949,"MULTIPOLYGON (((336335.234 1602955.393, 336334..."



📊 Statistiques pour la colonne 'Arbre' :


Arbre
0    420
1    331
Name: count, dtype: int64


   Nombre de parcelles SANS arbre ('Arbre' == 0) : 420


In [70]:
# Liste des colonnes que vous souhaitez conserver
colonnes_a_garder = [
    'parcel_id',
    'N°_PARCEL', 
    'TYP_SOL', 
    'Arbre', 
    'Long', 
    'Lat',
    'X_Centroid', 
    'Y_Centroid'
]

try:
    # Création d'un nouveau DataFrame avec uniquement ces colonnes
    df_selection = gdf_parcelles[colonnes_a_garder]

    print("✅ Sélection des colonnes effectuée avec succès !")
    
    # --- LIGNE AJOUTÉE --- #
    print(f"   Le nouveau DataFrame contient {df_selection.shape[0]} lignes et {df_selection.shape[1]} colonnes.")
    # --------------------- #
    
    print("\nAperçu du nouveau DataFrame (5 premières lignes) :")
    display(df_selection.head())

    # --- LIGNES AJOUTÉES --- #
    print("\nAperçu de 5 parcelles avec Arbre == 1 : 🌳")
    display(df_selection[df_selection['Arbre'] == 1].head())
    # ------------------------- #

except NameError:
    print("ERREUR : Le DataFrame 'gdf_parcelles' n'a pas été trouvé.")
except KeyError as e:
    print(f"ERREUR : La colonne {e} est introuvable dans le fichier.")
    print("Veuillez vérifier le nom exact des colonnes. Voici la liste des colonnes disponibles :")
    print(gdf_parcelles.columns.to_list())

✅ Sélection des colonnes effectuée avec succès !
   Le nouveau DataFrame contient 751 lignes et 8 colonnes.

Aperçu du nouveau DataFrame (5 premières lignes) :


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06
1,c6c85252-0ee6-44b2-97bf-99724c6dd923,147.0,Dior,0,0.000000,0.000000e+00,337365.773977,1.603611e+06
2,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06
3,c33f3753-34ef-4b61-ba17-3cc4fe29bbd3,149.0,Dekk,0,0.000000,0.000000e+00,336099.653300,1.603497e+06
4,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06



Aperçu de 5 parcelles avec Arbre == 1 : 🌳


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06
2,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06
4,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06
6,d28eb822-617a-4498-acbf-4f919ee1e3ae,144.0,Dior,1,335917.410954,1.602738e+06,335886.039020,1.602729e+06
10,2a2005cb-29a4-48bd-b7ad-fca2bc44db44,148.0,Dior,1,336894.131809,1.603453e+06,336894.334621,1.603456e+06


In [71]:
print(f"Nombre de lignes avant la suppression des doublons stricts : {len(df_selection)}")

# Supprimer les lignes qui sont des doublons sur l'ensemble des colonnes
df_selection.drop_duplicates(inplace=True)

print(f"Nombre de lignes après la suppression des doublons stricts : {len(df_selection)}")

Nombre de lignes avant la suppression des doublons stricts : 751
Nombre de lignes après la suppression des doublons stricts : 751


C:\Users\Cheikhou\AppData\Local\Temp\ipykernel_32936\812043785.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selection.drop_duplicates(inplace=True)


In [72]:
# Sépare le DataFrame en deux selon la condition sur "Arbre"
df_arbres = df_selection[df_selection['Arbre'] == 1].copy()
df_parcelles = df_selection[df_selection['Arbre'] != 1].copy()

print(f"Nombre de lignes avec 'Arbre' == 1: {len(df_arbres)}")
print(f"Nombre de lignes pour les parcelles : {len(df_parcelles)}")

Nombre de lignes avec 'Arbre' == 1: 331
Nombre de lignes pour les parcelles : 420


In [73]:
# Crée le GeoDataFrame pour les arbres
gdf_arbres = gpd.GeoDataFrame(
    df_arbres,
    geometry=gpd.points_from_xy(df_arbres.Long, df_arbres.Lat),
    crs="EPSG:4326"  # Coordonnées WGS84
)

print("GeoDataFrame pour les arbres créé avec succès.")
gdf_arbres.head()

GeoDataFrame pour les arbres créé avec succès.


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid,geometry
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06,POINT (337382.97662 1603607.33749)
2,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06,POINT (336145.75509 1603488.98846)
4,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06,POINT (336296.07434 1603033.94857)
6,d28eb822-617a-4498-acbf-4f919ee1e3ae,144.0,Dior,1,335917.410954,1.602738e+06,335886.039020,1.602729e+06,POINT (335917.41095 1602738.35263)
10,2a2005cb-29a4-48bd-b7ad-fca2bc44db44,148.0,Dior,1,336894.131809,1.603453e+06,336894.334621,1.603456e+06,POINT (336894.13181 1603453.182)


In [74]:
# Crée le GeoDataFrame pour les parcelles directement en WGS84
gdf_parcelles_wgs84 = gpd.GeoDataFrame(
    df_parcelles,
    geometry=gpd.points_from_xy(df_parcelles.X_Centroid, df_parcelles.Y_Centroid),
    crs="EPSG:4326"
)

print("GeoDataFrame pour les parcelles créé avec succès (CRS: EPSG:4326).")
gdf_parcelles_wgs84.head()

GeoDataFrame pour les parcelles créé avec succès (CRS: EPSG:4326).


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid,geometry
1,c6c85252-0ee6-44b2-97bf-99724c6dd923,147.0,Dior,0,0.0,0.0,337365.773977,1.603611e+06,POINT (337365.77398 1603611.29869)
3,c33f3753-34ef-4b61-ba17-3cc4fe29bbd3,149.0,Dekk,0,0.0,0.0,336099.653300,1.603497e+06,POINT (336099.6533 1603496.62979)
5,9ae158c8-ed7a-49cc-a548-fa876a904d75,150.0,Dior,0,0.0,0.0,336302.866304,1.603039e+06,POINT (336302.8663 1603039.27258)
7,a50e70b5-39dd-4412-9c9a-66781e618fc3,144.0,Dior,0,0.0,0.0,335886.039020,1.602729e+06,POINT (335886.03902 1602728.63143)
8,19d5192e-6e55-4af6-a7e2-558755bfce85,143.0,Dior,0,0.0,0.0,336334.936817,1.603098e+06,POINT (336334.93682 1603098.02693)


In [75]:
gdf_final = pd.concat([gdf_arbres, gdf_parcelles_wgs84], ignore_index=True)

# Le GeoDataFrame final est maintenant le résultat (remplace l'ancien gdf_wgs84)
gdf_wgs84 = gdf_final

print("Fusion terminée avec succès !")
print(f"CRS final : {gdf_wgs84.crs}")
print("Aperçu du GeoDataFrame final :")
display(gdf_wgs84.head())
print(f"\nNombre total de lignes : {len(gdf_wgs84)}")

Fusion terminée avec succès !
CRS final : EPSG:4326
Aperçu du GeoDataFrame final :


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid,geometry
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06,POINT (337382.97662 1603607.33749)
1,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06,POINT (336145.75509 1603488.98846)
2,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06,POINT (336296.07434 1603033.94857)
3,d28eb822-617a-4498-acbf-4f919ee1e3ae,144.0,Dior,1,335917.410954,1.602738e+06,335886.039020,1.602729e+06,POINT (335917.41095 1602738.35263)
4,2a2005cb-29a4-48bd-b7ad-fca2bc44db44,148.0,Dior,1,336894.131809,1.603453e+06,336894.334621,1.603456e+06,POINT (336894.13181 1603453.182)



Nombre total de lignes : 751


In [76]:
# Chemin vers le fichier contenant l'information 'Type_champ'
malou_csv_path = base_dir / "data" / "sols" / "csv" / "raw" / "malou_0_30.csv"

# Lecture du fichier CSV
df_malou = pd.read_csv(malou_csv_path)

# On harmonise le nom de la colonne clé avant la jointure.
df_malou.rename(columns={'N°_PARCELL': 'N°_PARCEL'}, inplace=True)

df_malou.head()

,Unnamed: 0,X_Centroid,Y_Centroid,N°_PARCEL,N°_FOYER,SAISON,NOM_UTILIS,NOM_PROPRI,UTL_2012,NOM_CHAMPS,...,Epaisseur_(cm),Superficie_(ha),Da_(g/cm3),Parcage,Couverture_sol,Antécédent_cultural,Termitère,Date,Saison,Kadd
0,0,337391.020695,1.603426e+06,1,1,1,Djimith FAYE,Djimith FAYE,0,Pifind,...,20,"1,21","1,56",1,2,2,0,8/6/2016,SS,2.0
1,1,336914.113937,1.603585e+06,15,1,1,Djimith FAYE,Djimith FAYE,0,Ngas nobenda,...,20,"0,38","1,53",0,0,2,0,8/6/2016,SS,1.0
2,2,334144.139128,1.604236e+06,2,1,1,Djimith FAYE,Djimith FAYE,0,Mbelpil 1,...,20,"0,27","1,55",0,0,2,1,8/6/2016,SS,1.0
3,3,333691.240137,1.603836e+06,3,1,1,Djimith FAYE,Djimith FAYE,0,Mbelpil 2 b,...,20,"0,35","1,64",0,2,2,1,8/6/2016,SS,0.0
4,4,333675.819844,1.603783e+06,4,1,1,Djimith FAYE,Djimith FAYE,0,Mbelpil 2 a,...,20,"0,45","1,64",0,2,2,1,8/6/2016,SS,0.0


In [77]:
# --- CORRECTION DES DONNÉES SOURCES ---

# 1. Correction du doublon (Parcelle 192) dans df_malou
print("--- Nettoyage de df_malou ---")
print(f"Lignes avant suppression des doublons : {len(df_malou)}")
df_malou.drop_duplicates(subset=['N°_PARCEL'], keep='first', inplace=True)
print(f"Lignes après suppression des doublons : {len(df_malou)}")


# 2. Suppression de la parcelle 201 dans gdf_wgs84
print("\n--- Nettoyage de gdf_wgs84 ---")
print(f"Lignes avant suppression de la parcelle 201 : {len(gdf_wgs84)}")
gdf_wgs84 = gdf_wgs84[gdf_wgs84['N°_PARCEL'] != 201].copy()
print(f"Lignes après suppression de la parcelle 201 : {len(gdf_wgs84)}")

--- Nettoyage de df_malou ---
Lignes avant suppression des doublons : 419
Lignes après suppression des doublons : 418

--- Nettoyage de gdf_wgs84 ---
Lignes avant suppression de la parcelle 201 : 751
Lignes après suppression de la parcelle 201 : 749


In [78]:
# Isoler les colonnes utiles du fichier de M. Malou
df_type_champ = df_malou[['N°_PARCEL', 'Type_champ']]

# --- AVANT LA JOINTURE ---
print("--- AVANT LA JOINTURE ---")
print(f"Dimensions de gdf_wgs84 : {gdf_wgs84.shape} (lignes, colonnes)")
# -------------------------

# --- JOINTURE ---
gdf_wgs84 = pd.merge(
    gdf_wgs84,
    df_type_champ,
    on='N°_PARCEL',
    how='left'
)
# ----------------

# --- APRÈS LA JOINTURE ---
print("\n--- APRÈS LA JOINTURE ---")
print(f"Dimensions de gdf_wgs84 : {gdf_wgs84.shape} (lignes, colonnes)")
# -----------------------

# --- VALIDATION ---
valeurs_manquantes = gdf_wgs84['Type_champ'].isnull().sum()

if valeurs_manquantes == 0:
    print("\n👍 Validation réussie : Toutes les parcelles ont trouvé une correspondance.")
else:
    print(f"\n🚨 ATTENTION : {valeurs_manquantes} parcelle(s) n'ont pas trouvé de correspondance.")

--- AVANT LA JOINTURE ---
Dimensions de gdf_wgs84 : (749, 9) (lignes, colonnes)

--- APRÈS LA JOINTURE ---
Dimensions de gdf_wgs84 : (749, 10) (lignes, colonnes)

👍 Validation réussie : Toutes les parcelles ont trouvé une correspondance.


In [79]:
import numpy as np

try:
    # On définit la présence d'arbre ('avec_arbr' / 'sans_arbr') en se basant sur la colonne 'Arbre'.
    presence_arbre = np.where(gdf_wgs84['Arbre'] == 1, 'avec_arbr', 'sans_arbr')

    # On s'assure que les autres colonnes de classification sont de type texte pour éviter les erreurs.
    gdf_wgs84['TYP_SOL'] = gdf_wgs84['TYP_SOL'].astype(str)
    gdf_wgs84['Type_champ'] = gdf_wgs84['Type_champ'].astype(str)

    # On crée la colonne 'type_ilot' en assemblant les bons composants.
    gdf_wgs84['type_ilot'] = (
        gdf_wgs84['TYP_SOL'] + '_' +
        gdf_wgs84['Type_champ'] + '_' +
        presence_arbre
    )

    # On convertit le résultat en minuscules pour la standardisation.
    gdf_wgs84['type_ilot'] = gdf_wgs84['type_ilot'].str.lower()
    
    # --- LIGNE AJOUTÉE --- #
    # On crée 'ZONE_PEDO' comme copie conforme de 'type_ilot'
    gdf_wgs84['ZONE_PEDO'] = gdf_wgs84['type_ilot']
    # --------------------- #

    print("✅ Colonnes 'type_ilot' et 'ZONE_PEDO' créées avec succès.")

    # On affiche un aperçu pour vérifier la logique.
    print("\nAperçu des colonnes de classification :")
    display(gdf_wgs84[['TYP_SOL', 'Type_champ', 'Arbre', 'type_ilot', 'ZONE_PEDO']].head())

except NameError:
    print("ERREUR : Le GeoDataFrame 'gdf_wgs84' n'a pas été trouvé.")
except KeyError as e:
    print(f"ERREUR : La colonne {e} est introuvable.")

✅ Colonnes 'type_ilot' et 'ZONE_PEDO' créées avec succès.

Aperçu des colonnes de classification :


,TYP_SOL,Type_champ,Arbre,type_ilot,ZONE_PEDO
0,Dior,CC,1,dior_cc_avec_arbr,dior_cc_avec_arbr
1,Dekk,CB,1,dekk_cb_avec_arbr,dekk_cb_avec_arbr
2,Dior,CB,1,dior_cb_avec_arbr,dior_cb_avec_arbr
3,Dior,CB,1,dior_cb_avec_arbr,dior_cb_avec_arbr
4,Dior,CB,1,dior_cb_avec_arbr,dior_cb_avec_arbr


In [80]:
gdf_wgs84.to_csv(output_points_path, sep=",", index=False)
print(f"Données enregistrées dans : {output_points_path}")

Données enregistrées dans : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\points_echantillonnage_complets.csv
